# Embed the Policy Summary

| Item | Description |
|---|---|
| Goal | Turn the Topic 68 policy summary from T3.5.5 into one unit-length embedding: the topic vector for Track 2. |
| Input | `output/T3.5.5/policy_summary.md`, the final summary text, and `policy_summary_meta.json` for its recorded token count. |
| Output | `output/T3.5.7/policy_summary_text.csv` (the text and its token count) and `output/T3.5.7/policy_summary_embedding.npy` (1 × 1024, float32). |
| Unit | One topic. |
| Model | `BAAI/bge-large-en-v1.5` from the project's local snapshot: the CLS token of the last layer, L2-normalized, at most 512 input tokens. Run on the GPU in float32, exactly as the definition vector (T3.5.1) and the statement embeddings (T3.5.3). |
| Variables | • `policy_summary_embedding`: $c_{\text{pol}} = e(\text{policy summary}) / \lVert e(\text{policy summary}) \rVert \in \mathbb{R}^{1024}$, where $e$ is the CLS embedding. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how this vector is used downstream.


## Step 1. Import packages

Read the summary, load the encoder, and save the vector.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from IPython.display import display


## Step 2. Define file locations

Read the summary and the model snapshot in place; save outputs in `output/T3.5.7/` under the TEMP folder.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.5.7"  # the identifier names this notebook's output folder
SUMMARY_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.5.5/policy_summary.md"
META_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.5.5/policy_summary_meta.json"
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--BAAI--bge-large-en-v1.5"
    "/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09"
)
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

TOPIC_ID = 68  # consumer data privacy legislation
MAX_TOKENS = 512  # the encoder's input limit
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.7


## Step 3. Load the policy summary

Read the text and the token count T3.5.5 recorded for it.


In [3]:
policy_summary = SUMMARY_FILE.read_text(encoding="utf-8").strip()
meta = json.loads(META_FILE.read_text(encoding="utf-8"))
assert policy_summary
assert len(policy_summary.split()) == meta["final_n_words"]
print(f"Policy summary: {meta['final_n_words']} words, {meta['final_n_bge_tokens']} bge tokens as recorded by T3.5.5")
print(policy_summary)


Policy summary: 221 words, 274 bge tokens as recorded by T3.5.5
The legislation imposes rigorous obligations on tech giants, broadband providers, healthcare organizations, drone operators, and financial institutions to secure consumer data and prevent its unauthorized sale or misuse. It grants individuals specific rights, including the ability to opt in or out of data collection, demand clear and conspicuous privacy policies, control their digital communications, and establish data "firewalls." Companies are required to implement "privacy by design," provide breach notifications within 72 hours, and ensure that sensitive information such as health records, social security numbers, and geolocation data is protected from discrimination and identity theft.

The regulatory scope targets a wide array of data practices, ranging from the deceptive monitoring of online activities and the unauthorized mapping of consumer movements to the commercialization of private data and the use of algorith

## Step 4. Load the encoder

Load the tokenizer and model from the local snapshot only, in evaluation mode on the GPU, and check the 512-token limit.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)
assert torch.cuda.is_available(), "run this notebook on a GPU node"
device = torch.device("cuda")
model = AutoModel.from_pretrained(MODEL_FOLDER, local_files_only=True).to(device).eval()
assert model.config.max_position_embeddings == MAX_TOKENS
print(f"Encoder: {MODEL_FOLDER.parent.parent.name}; hidden size {model.config.hidden_size}; input limit {MAX_TOKENS} tokens")
print(f"Device: {torch.cuda.get_device_name(device)}; dtype {next(model.parameters()).dtype}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoder: models--BAAI--bge-large-en-v1.5; hidden size 1024; input limit 512 tokens
Device: NVIDIA A40; dtype torch.float32


## Step 5. Count the tokens

Recount with the encoder's tokenizer and confirm the recorded count; the text must fit the input limit.


In [5]:
n_tokens = len(tokenizer(policy_summary)["input_ids"])
assert n_tokens == meta["final_n_bge_tokens"]
assert n_tokens <= MAX_TOKENS
print(f"Tokens including the special start and end tokens: {n_tokens}")


Tokens including the special start and end tokens: 274


## Step 6. Embed the summary

Run the encoder once, take the CLS token of the last layer, and scale it to unit length.


In [6]:
with torch.no_grad():
    encoded = tokenizer(policy_summary, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").to(device)
    last_layer = model(**encoded).last_hidden_state
    cls_vector = last_layer[:, 0]
    policy_summary_embedding = torch.nn.functional.normalize(cls_vector, dim=-1).cpu().numpy().astype(np.float32)
assert policy_summary_embedding.shape == (1, model.config.hidden_size)
assert abs(np.linalg.norm(policy_summary_embedding) - 1) < 1e-5
print(f"Shape: {policy_summary_embedding.shape}; norm: {np.linalg.norm(policy_summary_embedding):.6f}")
print(f"First five values: {np.round(policy_summary_embedding[0, :5], 4)}")


Shape: (1, 1024); norm: 1.000000
First five values: [-0.0379 -0.0295  0.0027  0.039   0.0234]


## Step 7. Save the text and the vector

Save the text with its token count beside the vector so a reader can see exactly what was embedded.


In [7]:
text_file = OUTPUT_FOLDER / "policy_summary_text.csv"
embedding_file = OUTPUT_FOLDER / "policy_summary_embedding.npy"
pd.DataFrame([{
    "topic_id": TOPIC_ID,
    "source_run_started": meta["run_started"],
    "policy_summary": policy_summary,
    "n_words": meta["final_n_words"],
    "n_tokens": n_tokens,
}]).to_csv(text_file, index=False)
np.save(embedding_file, policy_summary_embedding)
print(f"Saved: {text_file}")
print(f"Saved: {embedding_file}")
display(pd.read_csv(text_file)[["topic_id", "source_run_started", "n_words", "n_tokens"]])


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.7/policy_summary_text.csv
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.7/policy_summary_embedding.npy


,topic_id,source_run_started,n_words,n_tokens
0,68,2026-10-25T12:37:24+00:00,221,274
